NIYOJAN V3 — NOTEBOOK IMPLEMENTATION

In [1]:
from dotenv import load_dotenv
import os

In [14]:
import os
print(os.getcwd())

d:\Mini-Project\Gen AI\Niyojan-main\Agentic-HUB\Notebooks


In [19]:
load_dotenv("../../.env")

api_key = os.getenv("GEMINI_API_KEY")
print(api_key)

***


In [27]:
# ── Model choices ──────────────────────────────────────────────────────────
GEMINI_MODEL     = "gemini-flash-latest"   # Fast + cheap for reasoning
# ── Z-score for safety stock (1.65 = 95% service level) ───────────────────
Z_SCORE = 1.65

In [4]:
import pandas as pd
import numpy as np

In [5]:
%pwd

'd:\\Mini-Project\\Gen AI\\Niyojan-main\\Agentic-HUB\\Notebooks'

In [6]:
CSV_PATH = "forecast.csv"   # change to your file path

df = pd.read_csv(CSV_PATH)
print("Columns:", list(df.columns))
print(f"Products: {len(df)}")
df.head()

Columns: ['Product_ID', 'Product_Name', 'Category', 'Last_Week', 'Last_Week_Sales', 'Week_1_Forecast', 'Week_2_Forecast', 'Week_3_Forecast', 'Week_4_Forecast', 'Week_5_Forecast', 'Week_1_Final', 'Week_2_Final', 'Week_3_Final', 'Week_4_Final', 'Week_5_Final']
Products: 8


,Product_ID,Product_Name,Category,Last_Week,Last_Week_Sales,Week_1_Forecast,Week_2_Forecast,Week_3_Forecast,Week_4_Forecast,Week_5_Forecast,Week_1_Final,Week_2_Final,Week_3_Final,Week_4_Final,Week_5_Final
0,P001,Rice,Staples,2025-03-02,139,138,132,126,122,119,138,132,126,122,119
1,P002,Atta,Staples,2025-03-02,127,132,127,123,120,117,132,127,123,120,117
2,P003,Sugar,Staples,2025-03-02,92,91,93,96,97,99,91,93,96,97,99
3,P004,Sunflower Oil,Cooking Essentials,2025-03-02,139,122,120,118,116,114,122,120,118,116,114
4,P005,Milk,Dairy,2025-03-02,70,85,88,91,94,96,85,88,91,94,96


Preprocessing: Compute Derived Metrics

In [8]:
# ── Smart Default Constants ────────────────────────────────────────────────
# These are used for ALL products since the CSV doesn't contain this data.
# Change these values if you know your actual business defaults.
DEFAULT_LEAD_TIME   = 2    # weeks — standard FMCG replenishment lead time
DEFAULT_STOCK_MULT  = 2    # current_stock = mean_forecast × this multiplier


def preprocess_forecast(df: pd.DataFrame) -> list:
    """
    Takes the raw forecast CSV (any product IDs, any number of weeks)
    and returns a fully enriched product list ready for the agent.

    KNOWN LIMITATION:
    lead_time and current_stock are NOT in the CSV.
    We use smart defaults: lead_time=2 weeks, stock=2×mean_forecast.
    Simulation results are directionally correct, not warehouse-precise.
    """
    # Detect forecast columns dynamically — works for any number of weeks
    forecast_cols = [
        c for c in df.columns
        if c.startswith("Week_") and c.endswith("_Forecast")
    ]

    if not forecast_cols:
        raise ValueError("No 'Week_N_Forecast' columns found in CSV. Check your file format.")

    # ── Print warnings once upfront ────────────────────────────────────────
    print("=" * 60)
    print("NIYOJAN V3 — PREPROCESSING WARNINGS")
    print("=" * 60)
    print(f"  Forecast columns detected : {forecast_cols}")
    print(f"  Products found            : {len(df)}")
    print(f"  lead_time                 : DEFAULT = {DEFAULT_LEAD_TIME} weeks (not in CSV)")
    print(f"  current_stock             : DEFAULT = {DEFAULT_STOCK_MULT}× mean forecast (not in CSV)")
    print("  ➡  Simulation math will use these defaults.")
    print("  ➡  Results are approximate — connect to WMS for precision.")
    print("=" * 60)

    products = []

    for _, row in df.iterrows():
        pid = str(row["Product_ID"])

        # ── Extract weekly forecast values ──────────────────────────────────
        weekly_forecasts = [
            float(row[c]) for c in forecast_cols if not pd.isna(row[c])
        ]
        if not weekly_forecasts:
            print(f"{pid}: No valid forecast values found — skipping.")
            continue

        # ── Core values from CSV ────────────────────────────────────────────
        mean_demand     = float(np.mean(weekly_forecasts))
        week1_forecast  = weekly_forecasts[0]
        last_week_sales = float(row.get("Last_Week_Sales", mean_demand))

        # std_dev: use weekly spread if >1 week, else 10% of mean as floor
        std_dev = (
            float(np.std(weekly_forecasts))
            if len(weekly_forecasts) > 1
            else round(mean_demand * 0.10, 2)
        )
        # Prevent std_dev=0 edge case (all weeks identical forecast)
        if std_dev == 0:
            std_dev = round(mean_demand * 0.05, 2)

        # ── Smart defaults for missing fields ───────────────────────────────
        lead_time     = DEFAULT_LEAD_TIME
        current_stock = round(mean_demand * DEFAULT_STOCK_MULT)

        # ── Derived metrics (pure Python, zero LLM) ─────────────────────────
        volatility    = round(std_dev / mean_demand, 4) if mean_demand > 0 else 0
        safety_stock  = round(Z_SCORE * std_dev * np.sqrt(lead_time), 2)
        reorder_point = round((mean_demand * lead_time) + safety_stock, 2)
        inv_gap       = round(reorder_point - current_stock, 2)  # +ve = shortage

        # ── Risk classification — matches V2 decision engine logic ──────────
        ratio = week1_forecast / current_stock if current_stock > 0 else 999
        if ratio > 1.2:
            base_risk = "High"
        elif ratio < 0.5:
            base_risk = "Medium (Overstock)"
        else:
            base_risk = "Low"

        # ── Trend: first week vs last week of forecast ──────────────────────
        if len(weekly_forecasts) >= 2:
            trend_pct = round(
                ((weekly_forecasts[-1] - weekly_forecasts[0]) / weekly_forecasts[0]) * 100,
                2
            )
        else:
            trend_pct = 0.0

        products.append({
            # Identifiers
            "product_id"      : pid,
            "product_name"    : str(row.get("Product_Name", pid)),
            "category"        : str(row.get("Category", "Unknown")),
            # Raw values from CSV
            "last_week_sales" : last_week_sales,
            "week1_forecast"  : week1_forecast,
            "weekly_forecasts": weekly_forecasts,
            "mean_demand"     : round(mean_demand, 2),
            "std_dev"         : round(std_dev, 2),
            # Defaulted fields (not in CSV)
            "lead_time"       : lead_time,
            "current_stock"   : current_stock,
            "using_defaults"  : True,   # flag so agent/UI can show warning
            # Derived metrics
            "volatility"      : volatility,
            "safety_stock"    : safety_stock,
            "reorder_point"   : reorder_point,
            "inv_gap"         : inv_gap,
            "base_risk"       : base_risk,
            "trend_pct"       : trend_pct,
        })

    return products


# ── Run preprocessing — works on any CSV the forecast module generates ─────
FORECAST_DATA = preprocess_forecast(df)

# ── Preview table ──────────────────────────────────────────────────────────
print(f"\n{'ID':<6} {'Name':<22} {'Stock':>7} {'Gap':>8} {'Risk':<22} {'Volatility':>10}")
print("-" * 78)
for p in FORECAST_DATA:
    print(
        f"{p['product_id']:<6} {p['product_name']:<22} "
        f"{p['current_stock']:>7} {p['inv_gap']:>+8.1f} "
        f"{p['base_risk']:<22} {p['volatility']:>10.4f}"
    )

NIYOJAN V3 — PREPROCESSING WARNINGS
  Forecast columns detected : ['Week_1_Forecast', 'Week_2_Forecast', 'Week_3_Forecast', 'Week_4_Forecast', 'Week_5_Forecast']
  Products found            : 8
  lead_time                 : DEFAULT = 2 weeks (not in CSV)
  current_stock             : DEFAULT = 2× mean forecast (not in CSV)
  ➡  Simulation math will use these defaults.
  ➡  Results are approximate — connect to WMS for precision.

ID     Name                     Stock      Gap Risk                   Volatility
------------------------------------------------------------------------------
P001   Rice                       255    +15.8 Low                        0.0538
P002   Atta                       248    +11.9 Low                        0.0426
P003   Sugar                      190     +7.1 Medium (Overstock)         0.0300
P004   Sunflower Oil              236     +6.6 Low                        0.0240
P005   Milk                       182     +8.9 Medium (Overstock)         0.0437
P0

RAG: Ingest Report PDF into FAISS

In [ ]:
import os
import numpy as np
import faiss
import PyPDF2
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# ── Load environment variables ─────────────────────────────────────────────
load_dotenv("../../.env")  # adjust if needed

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found")

# ── Embedding Model (LOCAL, no API needed) ─────────────────────────────────
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# ── Upload your report PDF path here ──────────────────────────────────────
PDF_PATH = "niyojan_report.pdf"


def extract_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF file."""
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += (page.extract_text() or "") + "\n"
    return text.strip()


def chunk_text(text: str, chunk_size: int = 700, overlap: int = 100) -> list:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if len(c) > 50]


def embed_texts(texts: list) -> np.ndarray:
    """Generate embeddings using SentenceTransformers."""
    embeddings = embed_model.encode(texts)
    return np.array(embeddings, dtype="float32")


class FAISSVectorStore:
    def __init__(self):
        self.index = None
        self.chunks = []

    def build(self, chunks: list):
        print(f"Embedding {len(chunks)} chunks...")
        embeddings = embed_texts(chunks)

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

        self.chunks = chunks
        print(f"FAISS index built. dim={dim}, total_chunks={len(chunks)}")

    def search(self, query: str, top_k: int = 4) -> list:
        if self.index is None or len(self.chunks) == 0:
            return []

        q_emb = embed_texts([query])
        _, indices = self.index.search(q_emb, top_k)

        return [self.chunks[i] for i in indices[0] if i < len(self.chunks)]


# ── Build vector store from PDF ────────────────────────────────────────────
vector_store = FAISSVectorStore()

try:
    pdf_text = extract_pdf_text(PDF_PATH)
    chunks = chunk_text(pdf_text)

    print(f"PDF loaded: {len(pdf_text)} chars → {len(chunks)} chunks")

    vector_store.build(chunks)
    RAG_AVAILABLE = True

except FileNotFoundError:
    print(" PDF not found — upload required.")
    RAG_AVAILABLE = False

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2909.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PDF loaded: 1392 chars → 3 chunks
Embedding 3 chunks...
FAISS index built. dim=384, total_chunks=3


Agent State Definition

In [25]:
from typing import TypedDict, Optional, Annotated
import operator


class AgentState(TypedDict):
    # ── Inputs ────────────────────────────────────────────────────────────
    query            : str                   # User's natural language question
    forecast_data    : list                  # Preprocessed product list

    # ── Intent Detection ─────────────────────────────────────────────────
    intent           : str                   # retrieval | analysis | simulation | strategy
    simulation_percent: Optional[float]      # e.g. 15.0 if user said +15%

    # ── Execution Path Results ────────────────────────────────────────────
    rag_context      : list                  # Retrieved PDF chunks
    analysis_result  : dict                  # Computed metrics
    simulation_result: dict                  # Simulated scenario

    # ── LLM Input ─────────────────────────────────────────────────────────
    strategic_context: dict                  # Aggregated structured context

    # ── Final Output ──────────────────────────────────────────────────────
    final_response   : dict                  # Executive planning output


print("AgentState defined.")

AgentState defined.


Node 1: Intent Classification<br>
Classifies query into one of 4 intents. Uses Gemini at temperature=0 for determinism.

In [28]:
INTENT_PROMPT = """
You are an intent classifier for a supply chain planning AI.

Classify the user query into EXACTLY ONE of these 4 intents:

  retrieval  — User asks about the uploaded report, wants explanations from context/documents
  analysis   — User asks about inventory stats, volatility, risk rankings (no scenario change)
  simulation — User asks a "what if" scenario involving demand percentage change
  strategy   — User asks for a procurement plan, what to do, or general strategy

Also extract simulation_percent ONLY if intent=simulation.
  Example: "increase by 15%" → 15.0
  Example: "drop by 20%"    → -20.0
  If not applicable         → null

User Query: "{query}"

Respond with ONLY valid JSON, no markdown, no explanation:
{{"intent": "...", "simulation_percent": null}}
"""


def _fallback_classify(query: str) -> str:
    """Keyword-based fallback if Gemini fails to return valid JSON."""
    q = query.lower()
    if any(w in q for w in ["what if", "increase", "decrease", "scenario", "demand rise", "demand drop", "%"]):
        return "simulation"
    if any(w in q for w in ["report", "explain", "why", "what does", "tell me about"]):
        return "retrieval"
    if any(w in q for w in ["volatility", "gap", "metric", "highest", "lowest", "which product", "rank"]):
        return "analysis"
    return "strategy"


def _extract_percent(query: str) -> Optional[float]:
    """Extract first percentage number from query string."""
    match = re.search(r"([+-]?\d+(?:\.\d+)?)\s*%", query)
    if match:
        val = float(match.group(1))
        if any(w in query.lower() for w in ["drop", "decrease", "reduce", "fall", "decline"]):
            val = -abs(val)
        return val
    return None


def intent_node(state: AgentState) -> dict:
    """Node 1: Classify query intent and extract parameters."""
    query = state["query"]
    model = genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        generation_config=genai.GenerationConfig(temperature=0.0)
    )
    try:
        response = model.generate_content(INTENT_PROMPT.format(query=query))
        raw = response.text.strip()
        # Strip markdown fences Gemini sometimes adds
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        parsed = json.loads(raw)
        intent  = parsed.get("intent", "strategy")
        sim_pct = parsed.get("simulation_percent", None)
        if sim_pct is not None:
            sim_pct = float(sim_pct)
    except Exception as e:
        print(f"[IntentNode] Gemini parse failed ({e}), using fallback")
        intent  = _fallback_classify(query)
        sim_pct = _extract_percent(query)

    # Validate
    if intent not in {"retrieval", "analysis", "simulation", "strategy"}:
        intent = "strategy"

    print(f"[IntentNode] → intent={intent} | sim_pct={sim_pct}")
    return {"intent": intent, "simulation_percent": sim_pct}


def route_intent(state: AgentState) -> str:
    """Conditional edge: route to the correct execution path."""
    return state["intent"]   # maps to node names: retrieval / analysis / simulation / strategy


# Quick test
test_state = {"query": "What if demand increases by 15%?", "forecast_data": FORECAST_DATA,
              "intent": "", "simulation_percent": None, "rag_context": [],
              "analysis_result": {}, "simulation_result": {},
              "strategic_context": {}, "final_response": {}}
result = intent_node(test_state)
print(result)

[IntentNode] → intent=simulation | sim_pct=15.0
{'intent': 'simulation', 'simulation_percent': 15.0}


In [ ]:
import os
print(os.getcwd())

d:\Mini-Project\Gen AI\Niyojan-main\Agentic-HUB\Notebooks


Node 2: Retrieval (RAG Path)<br>
Used when user asks about the uploaded PDF report. No calculations here.

In [29]:
def retrieval_node(state: AgentState) -> dict:
    """
    Node 2 (Path 1): Retrieve relevant chunks from the PDF report using FAISS.
    Pure retrieval — zero calculations, zero LLM calls here.
    """
    query = state["query"]

    if not RAG_AVAILABLE:
        print("[RetrievalNode] No PDF loaded — returning empty context")
        return {"rag_context": ["No report PDF was uploaded. Please upload the Niyojan report PDF."]}

    chunks = vector_store.search(query, top_k=4)
    print(f"[RetrievalNode] Retrieved {len(chunks)} chunks")
    return {"rag_context": chunks}


# Test
test_state["query"] = "What does the report say about P005 and P007?"
r = retrieval_node(test_state)
print(f"Got {len(r['rag_context'])} chunks")

[RetrievalNode] Retrieved 4 chunks
Got 4 chunks


Node 3: Analysis (Descriptive Analytics Path): <br>Pure Python calculations — no LLM. Identifies highest volatility, biggest gaps, risk rankings.

In [30]:
def analysis_node(state: AgentState) -> dict:
    """
    Node 3 (Path 2): Compute descriptive analytics from forecast data.
    FULLY DETERMINISTIC — no LLM, no randomness.
    """
    data = state["forecast_data"]
    if not data:
        return {"analysis_result": {"error": "No forecast data available"}}

    # ── Sort by different metrics ──────────────────────────────────────────
    by_volatility = sorted(data, key=lambda x: x["volatility"], reverse=True)
    by_gap        = sorted(data, key=lambda x: x["inv_gap"], reverse=True)
    by_risk_score = sorted(data, key=lambda x: x["week1_forecast"] / max(x["current_stock"], 1), reverse=True)
    high_risk     = [p for p in data if p["base_risk"] == "High"]
    overstock     = [p for p in data if "Overstock" in p["base_risk"]]

    # ── Aggregate stats ────────────────────────────────────────────────────
    avg_volatility = round(np.mean([p["volatility"] for p in data]), 4)
    total_gap      = round(sum(p["inv_gap"] for p in data), 2)
    avg_lead_time  = round(np.mean([p["lead_time"] for p in data]), 1)

    result = {
        "total_products"     : len(data),
        "high_risk_count"    : len(high_risk),
        "overstock_count"    : len(overstock),
        "avg_volatility"     : avg_volatility,
        "total_inventory_gap": total_gap,
        "avg_lead_time_weeks": avg_lead_time,

        # Top-3 rankings
        "highest_volatility_skus": [
            {"id": p["product_id"], "name": p["product_name"], "volatility": p["volatility"]}
            for p in by_volatility[:3]
        ],
        "largest_gap_skus": [
            {"id": p["product_id"], "name": p["product_name"],
             "gap": p["inv_gap"], "rop": p["reorder_point"], "stock": p["current_stock"]}
            for p in by_gap[:3]
        ],
        "high_risk_skus": [
            {"id": p["product_id"], "name": p["product_name"],
             "forecast": p["week1_forecast"], "stock": p["current_stock"], "risk": p["base_risk"]}
            for p in by_risk_score[:5]
        ],
        "overstock_skus": [
            {"id": p["product_id"], "name": p["product_name"], "stock": p["current_stock"]}
            for p in overstock
        ],
    }

    print(f"[AnalysisNode] High risk={len(high_risk)} | Overstock={len(overstock)} | AvgVolatility={avg_volatility}")
    return {"analysis_result": result}


# Test
test_state["forecast_data"] = FORECAST_DATA
ar = analysis_node(test_state)
print(json.dumps(ar["analysis_result"], indent=2))

[AnalysisNode] High risk=0 | Overstock=4 | AvgVolatility=0.0341
{
  "total_products": 8,
  "high_risk_count": 0,
  "overstock_count": 4,
  "avg_volatility": 0.0341,
  "total_inventory_gap": 66.26,
  "avg_lead_time_weeks": 2.0,
  "highest_volatility_skus": [
    {
      "id": "P007",
      "name": "Soft Drinks",
      "volatility": 0.0581
    },
    {
      "id": "P001",
      "name": "Rice",
      "volatility": 0.0538
    },
    {
      "id": "P005",
      "name": "Milk",
      "volatility": 0.0437
    }
  ],
  "largest_gap_skus": [
    {
      "id": "P001",
      "name": "Rice",
      "gap": 15.8,
      "rop": 270.8,
      "stock": 255
    },
    {
      "id": "P002",
      "name": "Atta",
      "gap": 11.89,
      "rop": 259.89,
      "stock": 248
    },
    {
      "id": "P007",
      "name": "Soft Drinks",
      "gap": 11.38,
      "rop": 182.38,
      "stock": 171
    }
  ],
  "high_risk_skus": [
    {
      "id": "P001",
      "name": "Rice",
      "forecast": 138.0,
      "stock

Node 4: Simulation (Counterfactual Engine)<br>
The most important path. Handles "what if demand increases by X%" scenarios.
**Fully deterministic — no LLM involved in any calculation.**

In [31]:
def _classify_gap_risk(gap: float) -> str:
    """Risk classification based on inventory gap after simulation."""
    if gap > 200:
        return "High"
    elif gap > 100:
        return "Medium"
    elif gap > 0:
        return "Low"
    else:
        return "Safe (Stock Sufficient)"


def simulation_node(state: AgentState) -> dict:
    """
    Node 4 (Path 3): Counterfactual demand scenario simulation.
    Recalculates safety stock, reorder point, inventory gap, and risk
    for each product under the simulated demand change.

    Formula:
      new_demand    = forecast_demand × (1 + percent/100)
      safety_stock  = Z × std_dev × √lead_time
      ROP           = (new_demand × lead_time) + safety_stock
      gap           = ROP - current_stock
    """
    data    = state["forecast_data"]
    pct     = state.get("simulation_percent") or 0.0
    factor  = 1 + (pct / 100)

    simulated_products = []

    for p in data:
        new_demand    = round(p["mean_demand"] * factor, 2)
        safety_stock  = round(Z_SCORE * p["std_dev"] * np.sqrt(p["lead_time"]), 2)
        new_rop       = round((new_demand * p["lead_time"]) + safety_stock, 2)
        new_gap       = round(new_rop - p["current_stock"], 2)
        new_risk      = _classify_gap_risk(new_gap)
        demand_change = round(new_demand - p["mean_demand"], 2)

        simulated_products.append({
            "product_id"       : p["product_id"],
            "product_name"     : p["product_name"],
            "category"         : p["category"],
            "base_demand"      : p["mean_demand"],
            "new_demand"       : new_demand,
            "demand_change"    : demand_change,
            "current_stock"    : p["current_stock"],
            "safety_stock"     : safety_stock,
            "new_rop"          : new_rop,
            "new_gap"          : new_gap,
            "base_risk"        : p["base_risk"],
            "simulated_risk"   : new_risk,
            "risk_escalated"   : (new_risk == "High" and p["base_risk"] != "High"),
        })

    # Summary stats
    high_risk_sim  = [p for p in simulated_products if p["simulated_risk"] == "High"]
    newly_at_risk  = [p for p in simulated_products if p["risk_escalated"]]
    total_gap_sim  = round(sum(p["new_gap"] for p in simulated_products), 2)

    result = {
        "scenario"            : f"{'+' if pct >= 0 else ''}{pct}% demand change",
        "simulation_factor"   : factor,
        "high_risk_count"     : len(high_risk_sim),
        "newly_at_risk_count" : len(newly_at_risk),
        "total_simulated_gap" : total_gap_sim,
        "high_risk_products"  : sorted(high_risk_sim, key=lambda x: x["new_gap"], reverse=True),
        "newly_at_risk"       : newly_at_risk,
        "all_products"        : simulated_products,
    }

    print(f"[SimulationNode] Scenario={result['scenario']} | "
          f"HighRisk={len(high_risk_sim)} | NewlyAtRisk={len(newly_at_risk)}")
    return {"simulation_result": result}


# Test with +15%
test_state["simulation_percent"] = 15.0
sr = simulation_node(test_state)
print(f"\nScenario: {sr['simulation_result']['scenario']}")
for p in sr["simulation_result"]["high_risk_products"]:
    print(f"  {p['product_name']:20s} gap={p['new_gap']:+.1f} risk={p['simulated_risk']}")

[SimulationNode] Scenario=+15.0% demand change | HighRisk=0 | NewlyAtRisk=0

Scenario: +15.0% demand change


Node 5: Strategy Context Builder<br>

Aggregates all execution path results into a clean structured context.
This is the **bridge between data and LLM**. The LLM never sees raw numbers directly.

In [32]:
def strategy_node(state: AgentState) -> dict:
    """
    Node 5: Aggregates all path results into structured strategic context.
    This context is what the LLM will reason over — structured, not raw CSV data.
    """
    intent           = state.get("intent", "strategy")
    forecast_data    = state.get("forecast_data", [])
    analysis_result  = state.get("analysis_result", {})
    simulation_result= state.get("simulation_result", {})
    rag_context      = state.get("rag_context", [])
    query            = state.get("query", "")

    # ── Always compute base analysis if not already done ───────────────────
    if not analysis_result and forecast_data:
        analysis_result = analysis_node(state)["analysis_result"]

    # ── Build top-risk summary from raw data ───────────────────────────────
    high_risk = sorted(
        [p for p in forecast_data if p["base_risk"] == "High"],
        key=lambda x: x["inv_gap"], reverse=True
    )
    top_volatile = sorted(forecast_data, key=lambda x: x["volatility"], reverse=True)[:3]

    strategic_context = {
        "user_query"      : query,
        "intent"          : intent,

        # Core metrics
        "summary_metrics" : {
            "total_products"   : len(forecast_data),
            "high_risk_count"  : len(high_risk),
            "avg_volatility"   : analysis_result.get("avg_volatility"),
            "total_gap"        : analysis_result.get("total_inventory_gap"),
        },

        # Prioritized SKUs
        "high_risk_skus"  : [
            {
                "id"       : p["product_id"],
                "name"     : p["product_name"],
                "category" : p["category"],
                "gap"      : p["inv_gap"],
                "stock"    : p["current_stock"],
                "forecast" : p["week1_forecast"],
                "volatility": p["volatility"],
                "lead_time": p["lead_time"],
                "risk"     : p["base_risk"],
            }
            for p in high_risk[:5]
        ],

        # Volatile SKUs (demand unpredictability)
        "volatile_skus"   : [
            {"id": p["product_id"], "name": p["product_name"], "volatility": p["volatility"]}
            for p in top_volatile
        ],

        # Simulation context (populated only if simulation ran)
        "simulation"      : {
            "ran"                : bool(simulation_result),
            "scenario"           : simulation_result.get("scenario", "N/A"),
            "high_risk_count"    : simulation_result.get("high_risk_count", 0),
            "newly_at_risk_count": simulation_result.get("newly_at_risk_count", 0),
            "total_gap"          : simulation_result.get("total_simulated_gap", 0),
            "top_impacted"       : simulation_result.get("high_risk_products", [])[:4],
        },

        # RAG context (populated only if retrieval ran)
        "report_context"  : rag_context[:3] if rag_context else [],
    }

    print(f"[StrategyNode] Context built | intent={intent} | "
          f"high_risk={len(high_risk)} | sim_ran={bool(simulation_result)}")
    return {"strategic_context": strategic_context}


# Test
test_state["analysis_result"] = ar["analysis_result"]
test_state["simulation_result"] = sr["simulation_result"]
test_state["intent"] = "simulation"
sc = strategy_node(test_state)
print(json.dumps(sc["strategic_context"]["summary_metrics"], indent=2))

[StrategyNode] Context built | intent=simulation | high_risk=0 | sim_ran=True
{
  "total_products": 8,
  "high_risk_count": 0,
  "avg_volatility": 0.0341,
  "total_gap": 66.26
}


Node 6: Strategic LLM Reasoning<br>

**The ONLY place the LLM is called.** It receives structured context — never raw CSV data.
It does reasoning and language, not math.

In [33]:
STRATEGIC_SYSTEM_PROMPT = """
You are NIYOJAN — an expert Supply Chain Planning AI.

You receive STRUCTURED DATA from deterministic calculations. Your role is:
  ✅ Synthesize data into strategic insights
  ✅ Prioritize SKUs by risk and urgency
  ✅ Generate actionable 4-week procurement plans
  ✅ Explain WHY risks exist in business terms
  ❌ NEVER perform calculations yourself
  ❌ NEVER invent numbers not present in the input
  ❌ NEVER guess or hallucinate inventory data

Respond in structured JSON matching this schema:
{
  "executive_summary": "2-3 sentence overview of the situation",
  "high_risk_products": [
    {"product": "name", "risk_level": "High/Medium/Low", "reason": "why risky", "urgency": "Immediate/Soon/Monitor"}
  ],
  "key_risk_drivers": ["driver 1", "driver 2", "driver 3"],
  "recommended_actions": [
    {"action": "specific action", "target_skus": ["P001"], "priority": "High/Medium"}
  ],
  "four_week_plan": {
    "week_1": "actions for week 1",
    "week_2": "actions for week 2",
    "week_3": "actions for week 3",
    "week_4": "actions for week 4"
  },
  "monitoring_strategy": "what to track and how often"
}
"""

USER_PROMPT_TEMPLATE = """
User Query: {query}

Structured Context:
{context_json}

Generate a strategic executive planning response.
"""


def reasoning_node(state: AgentState) -> dict:
    """
    Node 6: Strategic LLM Reasoning.
    Receives structured context, returns structured executive output.
    """
    context = state.get("strategic_context", {})
    query   = state.get("query", "Give me a strategic procurement plan")

    # Serialize context to JSON string for LLM
    context_json = json.dumps(context, indent=2)

    model = genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        system_instruction=STRATEGIC_SYSTEM_PROMPT,
        generation_config=genai.GenerationConfig(
            temperature=0.2,   # Low temp = consistent, factual outputs
            response_mime_type="application/json"  # Force JSON output
        )
    )

    prompt = USER_PROMPT_TEMPLATE.format(
        query=query,
        context_json=context_json
    )

    try:
        response = model.generate_content(prompt)
        raw = response.text.strip()
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        result = json.loads(raw)
    except Exception as e:
        print(f"[ReasoningNode] Parse error: {e}")
        result = {
            "executive_summary": "System encountered an error generating insights.",
            "high_risk_products": [],
            "key_risk_drivers": [str(e)],
            "recommended_actions": [],
            "four_week_plan": {"week_1": "", "week_2": "", "week_3": "", "week_4": ""},
            "monitoring_strategy": ""
        }

    print(f"[ReasoningNode] ✅ Response generated")
    return {"final_response": result}

print("ReasoningNode defined.")

ReasoningNode defined.


Node 7: Executive Formatter<br>

Adds metadata and formats the final output for display.

In [35]:
from datetime import datetime


def formatter_node(state: AgentState) -> dict:
    """
    Node 7: Executive Formatter.
    Enriches the LLM output with metadata and ensures consistent structure.
    """
    raw_response = state.get("final_response", {})
    context      = state.get("strategic_context", {})

    formatted = {
        # Metadata
        "generated_at"     : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "query"            : state.get("query"),
        "intent"           : state.get("intent"),
        "simulation_scenario": context.get("simulation", {}).get("scenario", None),

        # Executive content
        "executive_summary": raw_response.get("executive_summary", ""),
        "high_risk_products": raw_response.get("high_risk_products", []),
        "key_risk_drivers" : raw_response.get("key_risk_drivers", []),
        "recommended_actions": raw_response.get("recommended_actions", []),
        "four_week_plan"   : raw_response.get("four_week_plan", {}),
        "monitoring_strategy": raw_response.get("monitoring_strategy", ""),
    }

    return {"final_response": formatted}


def pretty_print_output(output: dict):
    """Human-readable display of the final agent output."""
    print("\n" + "="*70)
    print(" NIYOJAN V3 — EXECUTIVE PLANNING OUTPUT")
    print("="*70)
    print(f" Generated: {output.get('generated_at')}")
    print(f" Query    : {output.get('query')}")
    print(f" Intent   : {output.get('intent')}")
    if output.get('simulation_scenario'):
        print(f"🔬 Scenario : {output.get('simulation_scenario')}")

    print("\n EXECUTIVE SUMMARY")
    print("-"*50)
    print(output.get('executive_summary', ''))

    print("\n  HIGH-RISK PRODUCTS")
    print("-"*50)
    for p in output.get('high_risk_products', []):
        print(f"  • {p.get('product')} [{p.get('risk_level')}] — {p.get('reason')} | Urgency: {p.get('urgency')}")

    print("\n KEY RISK DRIVERS")
    print("-"*50)
    for d in output.get('key_risk_drivers', []):
        print(f"  • {d}")

    print("\n RECOMMENDED ACTIONS")
    print("-"*50)
    for a in output.get('recommended_actions', []):
        skus = ', '.join(a.get('target_skus', []))
        print(f"  [{a.get('priority')}] {a.get('action')} → SKUs: {skus}")

    print("\n 4-WEEK ACTION PLAN")
    print("-"*50)
    plan = output.get('four_week_plan', {})
    for week in ['week_1', 'week_2', 'week_3', 'week_4']:
        print(f"  {week.replace('_', ' ').upper()}: {plan.get(week, '')}")

    print("\n MONITORING STRATEGY")
    print("-"*50)
    print(output.get('monitoring_strategy', ''))
    print("="*70)


print("Formatter defined.")

Formatter defined.


LangGraph Implementation

In [36]:
from langgraph.graph import StateGraph, START, END


# ── Build the graph ────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Add all nodes
builder.add_node("intent_classification", intent_node)
builder.add_node("retrieval"            , retrieval_node)
builder.add_node("analysis"             , analysis_node)
builder.add_node("simulation"           , simulation_node)
builder.add_node("strategy_context"     , strategy_node)
builder.add_node("llm_reasoning"        , reasoning_node)
builder.add_node("formatter"            , formatter_node)

# ── Edges ──────────────────────────────────────────────────────────────────
# Entry point
builder.add_edge(START, "intent_classification")

# Conditional routing based on intent
builder.add_conditional_edges(
    "intent_classification",
    route_intent,
    {
        "retrieval" : "retrieval",
        "analysis"  : "analysis",
        "simulation": "simulation",
        "strategy"  : "strategy_context",  # strategy goes directly to context builder
    }
)

# All execution paths converge at strategy_context
builder.add_edge("retrieval" , "strategy_context")
builder.add_edge("analysis"  , "strategy_context")
builder.add_edge("simulation", "strategy_context")

# Context → LLM → Formatter → End
builder.add_edge("strategy_context", "llm_reasoning")
builder.add_edge("llm_reasoning"   , "formatter")
builder.add_edge("formatter"       , END)

# Compile
agent = builder.compile()

print("✅ LangGraph agent compiled successfully!")
print("\nGraph structure:")
print("  START")
print("    ↓")
print("  intent_classification")
print("    ↓ (conditional)")
print("  [retrieval | analysis | simulation | strategy_context]")
print("    ↓ (all converge)")
print("  strategy_context → llm_reasoning → formatter → END")

✅ LangGraph agent compiled successfully!

Graph structure:
  START
    ↓
  intent_classification
    ↓ (conditional)
  [retrieval | analysis | simulation | strategy_context]
    ↓ (all converge)
  strategy_context → llm_reasoning → formatter → END


In [37]:
def run_agent(query: str, forecast_data: list = None, verbose: bool = True) -> dict:
    """
    Main entry point for the NIYOJAN V3 Agentic Hub.

    Args:
        query        : Natural language planning question from the user
        forecast_data: Preprocessed product list (from preprocess_forecast)
        verbose      : Whether to print the formatted output

    Returns:
        Final structured response dict
    """
    if forecast_data is None:
        forecast_data = FORECAST_DATA

    # Initialize state with defaults for all keys
    initial_state: AgentState = {
        "query"             : query,
        "forecast_data"     : forecast_data,
        "intent"            : "",
        "simulation_percent": None,
        "rag_context"       : [],
        "analysis_result"   : {},
        "simulation_result" : {},
        "strategic_context" : {},
        "final_response"    : {},
    }

    result = agent.invoke(initial_state)
    output = result["final_response"]

    if verbose:
        pretty_print_output(output)

    return output


print("run_agent() ready.")

run_agent() ready.


Test 1: Simulation Query

In [38]:
result_1 = run_agent("What if demand increases by 15% next month?")

[IntentNode] → intent=simulation | sim_pct=15.0
[SimulationNode] Scenario=+15.0% demand change | HighRisk=0 | NewlyAtRisk=0
[AnalysisNode] High risk=0 | Overstock=4 | AvgVolatility=0.0341
[StrategyNode] Context built | intent=simulation | high_risk=0 | sim_ran=True
[ReasoningNode] ✅ Response generated

 NIYOJAN V3 — EXECUTIVE PLANNING OUTPUT
 Generated: 2026-03-23 16:28:58
 Query    : What if demand increases by 15% next month?
 Intent   : simulation
🔬 Scenario : +15.0% demand change

 EXECUTIVE SUMMARY
--------------------------------------------------
A 15% demand surge next month is manageable with zero products currently projected to reach high-risk status. However, the total inventory gap will expand significantly from 66.26 to 323.3 units, necessitating a proactive procurement shift to maintain service levels across the portfolio.

  HIGH-RISK PRODUCTS
--------------------------------------------------

 KEY RISK DRIVERS
--------------------------------------------------
  • Sign

In [39]:
result_2 = run_agent("What should I do for procurement next month?")

[IntentNode] → intent=strategy | sim_pct=None
[AnalysisNode] High risk=0 | Overstock=4 | AvgVolatility=0.0341
[StrategyNode] Context built | intent=strategy | high_risk=0 | sim_ran=False
[ReasoningNode] ✅ Response generated

 NIYOJAN V3 — EXECUTIVE PLANNING OUTPUT
 Generated: 2026-03-23 16:30:18
 Query    : What should I do for procurement next month?
 Intent   : strategy
🔬 Scenario : N/A

 EXECUTIVE SUMMARY
--------------------------------------------------
The supply chain is currently in a stable state with no immediate high-risk stockouts identified across the 8 monitored products. Procurement efforts for the next month should focus on closing the total inventory gap of 66.26 units while proactively managing moderate demand volatility in the Soft Drinks and Rice categories.

  HIGH-RISK PRODUCTS
--------------------------------------------------
  • Soft Drinks [Medium] — Highest demand volatility (0.0581) in the portfolio, requiring closer monitoring of consumption patterns. | Urg

In [40]:
result_3 = run_agent("Which products have the highest inventory risk right now?")

[IntentNode] → intent=analysis | sim_pct=None
[AnalysisNode] High risk=0 | Overstock=4 | AvgVolatility=0.0341
[StrategyNode] Context built | intent=analysis | high_risk=0 | sim_ran=False
[ReasoningNode] ✅ Response generated

 NIYOJAN V3 — EXECUTIVE PLANNING OUTPUT
 Generated: 2026-03-23 16:30:41
 Query    : Which products have the highest inventory risk right now?
 Intent   : analysis
🔬 Scenario : N/A

 EXECUTIVE SUMMARY
--------------------------------------------------
The current inventory status is stable with no products flagged as high risk for immediate stockout. However, attention is required for Soft Drinks, Rice, and Milk, which exhibit the highest demand volatility and contribute to a total inventory gap of 66.26 units.

  HIGH-RISK PRODUCTS
--------------------------------------------------
  • Soft Drinks (P007) [Medium] — Highest demand volatility in the portfolio at 0.0581. | Urgency: Monitor
  • Rice (P001) [Medium] — Significant demand volatility of 0.0538 affecting st

In [41]:
result_4 = run_agent("What does the report say about P005 and P007 stock-out risk?")

[IntentNode] → intent=retrieval | sim_pct=None
[RetrievalNode] Retrieved 4 chunks
[AnalysisNode] High risk=0 | Overstock=4 | AvgVolatility=0.0341
[StrategyNode] Context built | intent=retrieval | high_risk=0 | sim_ran=False
[ReasoningNode] ✅ Response generated

 NIYOJAN V3 — EXECUTIVE PLANNING OUTPUT
 Generated: 2026-03-23 16:31:13
 Query    : What does the report say about P005 and P007 stock-out risk?
 Intent   : retrieval
🔬 Scenario : N/A

 EXECUTIVE SUMMARY
--------------------------------------------------
P005 (Milk) and P007 (Soft Drinks) are currently flagged for potential stock-out risks due to significant demand acceleration. Both products have shown a sharp increase in forecasted units compared to the previous week, necessitating immediate inventory review.

  HIGH-RISK PRODUCTS
--------------------------------------------------
  • P007 (Soft Drinks) [High] — High demand acceleration (78 vs 56 last week) and highest volatility (0.0581) in the portfolio. | Urgency: Immediate

In [42]:
result_5 = run_agent("What if demand drops by 20%? Which products will be overstocked?")

[IntentNode] → intent=simulation | sim_pct=-20.0
[SimulationNode] Scenario=-20.0% demand change | HighRisk=0 | NewlyAtRisk=0
[AnalysisNode] High risk=0 | Overstock=4 | AvgVolatility=0.0341
[StrategyNode] Context built | intent=simulation | high_risk=0 | sim_ran=True
[ReasoningNode] ✅ Response generated

 NIYOJAN V3 — EXECUTIVE PLANNING OUTPUT
 Generated: 2026-03-23 16:31:38
 Query    : What if demand drops by 20%? Which products will be overstocked?
 Intent   : simulation
🔬 Scenario : -20.0% demand change

 EXECUTIVE SUMMARY
--------------------------------------------------
A 20% reduction in demand shifts the supply chain risk from stockouts to significant overstocking, resulting in a projected surplus of 276.46 units across the portfolio. While no products currently face critical stockout risks, the primary challenge is managing excess inventory and preventing capital tie-up.

  HIGH-RISK PRODUCTS
--------------------------------------------------
  • Soft Drinks [Medium] — Highest 